# Pattern 01 — Decision capture

What this shows: `@capture` auto-records a function's inputs, outputs, timing,
and errors. The resulting `DecisionSnapshot` is the audit atom — what the
system DID, immutably, for every invocation.
When to reach for it: any SDK-instrumented function where you need an audit
trail of every call (who asked, what answer, how long, what model, what version).
`@capture` works on both sync and async functions; this pattern uses sync for
portability between scripts and notebooks.
See also: patterns/06_examiner_bundle.py (reproducibility layer on top of capture)

In [ ]:
import briefcase
from briefcase.decorators import capture

# Route captured records to an in-memory exporter so we can inspect them
# ("console" prints to stderr; a "*.jsonl" path appends to a file).
recorder = briefcase.observe("memory")

# async_capture=False records synchronously so we can inspect immediately.
@capture(decision_type="sentiment_classification", context_version="v1", async_capture=False)
def classify(document: str, model: str = "gpt-4o") -> dict:
    """Dummy 'LLM' — deterministic so the pattern runs offline."""
    score = 0.92 if "great" in document.lower() else 0.31
    return {"label": "positive" if score > 0.5 else "negative", "score": score}

## Invoke the instrumented function

In [ ]:
# Every @capture'd call produces a decision record: inputs, outputs,
# started_at, ended_at, execution_time_ms, context_version, and any error
# raised — each keyed by a fresh decision_id (UUID).
result = classify("The product is great", model="gpt-4o")
print(f"Classifier returned: {result}")

## A second call produces a separate snapshot

In [ ]:
# Each invocation produces its own immutable record — useful for
# comparing runs, reproducing a specific call, or building drift metrics.
result2 = classify("This is terrible", model="gpt-4o")
print(f"Second call:         {result2}")

# Inspect the captured audit atoms collected by the exporter.
print(f"\nCaptured {len(recorder.records)} decision record(s):")
for rec in recorder.records:
    print(f"  - {rec['decision_type']} ({rec['execution_time_ms']:.3f} ms): "
          f"inputs={rec['inputs']} -> outputs={rec['outputs']}")

## Errors are captured too

In [ ]:
# If the wrapped function raises, @capture records the exception on the
# record and re-raises it. The audit trail retains failed calls with
# full context — important for debugging and post-mortem analysis.
@capture(decision_type="always_fails", async_capture=False)
def always_fails(x: int) -> int:
    raise ValueError(f"no: {x}")

try:
    always_fails(42)
except ValueError as e:
    print(f"\nCaptured error: {type(e).__name__}: {e}")
    print(f"Total records (including the failed call): {len(recorder.records)}")